# United States Top 50 Playlist Analytics — Data Validation

## Objective

This notebook validates and prepares the historical United States Top 50 playlist dataset for downstream exploratory analysis.

The validation process checks data types, date coverage, ranking integrity, missing values, duplicate records, and snapshot consistency.

## 1. Data Ingestion

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/Atlantic_United_States.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully!
Rows: 27,800
Columns: 10


## 2. Initial Dataset Inspection

In [2]:
df.head()

,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url
0,18-05-2024,1,Ella Baila Sola,Eslabon Armado,89,165671,album,16,False,https://i.scdn.co/image/ab67616d0000b273dfddf1...
1,18-05-2024,2,Last Night,Morgan Wallen,89,163854,album,36,True,https://i.scdn.co/image/ab67616d0000b273705079...
2,18-05-2024,3,All My Life (feat. J. Cole),Lil Durk & J. Cole,84,223878,single,1,True,https://i.scdn.co/image/ab67616d0000b2737c173b...
3,18-05-2024,4,un x100to,Grupo Frontera & Bad Bunny,99,194563,single,1,False,https://i.scdn.co/image/ab67616d0000b273716c0b...
4,18-05-2024,5,Kill Bill,SZA,94,153946,album,23,False,https://i.scdn.co/image/ab67616d0000b2730c471c...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27800 entries, 0 to 27799
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   date             27800 non-null  object
 1   position         27800 non-null  int64 
 2   song             27800 non-null  object
 3   artist           27800 non-null  object
 4   popularity       27800 non-null  int64 
 5   duration_ms      27800 non-null  int64 
 6   album_type       27800 non-null  object
 7   total_tracks     27800 non-null  int64 
 8   is_explicit      27800 non-null  bool  
 9   album_cover_url  27800 non-null  object
dtypes: bool(1), int64(4), object(5)
memory usage: 1.9+ MB


In [4]:
df.describe(include="all")

,date,position,song,artist,popularity,duration_ms,album_type,total_tracks,is_explicit,album_cover_url
count,27800,27800.000000,27800,27800,27800.000000,27800.000000,27800,27800.000000,27800,27800
unique,555,NaN,943,297,NaN,NaN,3,NaN,2,538
top,01-03-2025,NaN,Something in the Orange,Taylor Swift,NaN,NaN,album,NaN,False,https://i.scdn.co/image/ab67616d0000b273705079...
freq,100,NaN,537,2016,NaN,NaN,19742,NaN,14465,1315
mean,NaN,25.500000,NaN,NaN,87.665612,198635.820180,NaN,12.945000,NaN,NaN
std,NaN,14.431129,NaN,NaN,10.463848,48057.863668,NaN,9.457927,NaN,NaN
min,NaN,1.000000,NaN,NaN,0.000000,37314.000000,NaN,1.000000,NaN,NaN
25%,NaN,13.000000,NaN,NaN,85.000000,166453.000000,NaN,2.000000,NaN,NaN
50%,NaN,25.500000,NaN,NaN,89.000000,189386.000000,NaN,14.000000,NaN,NaN
75%,NaN,38.000000,NaN,NaN,93.000000,228013.000000,NaN,18.000000,NaN,NaN


## 3. Data Quality Validation

In [5]:
df["date"] = pd.to_datetime(
    df["date"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("Minimum date:", df["date"].min())
print("Maximum date:", df["date"].max())

Minimum date: 2024-05-18 00:00:00
Maximum date: 2025-11-27 00:00:00


In [6]:
print("Minimum position:", df["position"].min())
print("Maximum position:", df["position"].max())

invalid_positions = df[
    ~df["position"].between(1, 50)
]

print("Invalid positions:", len(invalid_positions))

Minimum position: 1
Maximum position: 50
Invalid positions: 0


In [7]:
missing_values = df.isnull().sum()

print(missing_values)

date               0
position           0
song               0
artist             0
popularity         0
duration_ms        0
album_type         0
total_tracks       0
is_explicit        0
album_cover_url    0
dtype: int64


## 4. Duplicate and Snapshot Investigation

In [8]:
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 10


In [10]:
song_date_duplicates = (
    df.groupby(["date", "song"])
    .size()
    .reset_index(name="count")
)

song_date_duplicates = song_date_duplicates[
    song_date_duplicates["count"] > 1
]

print("Duplicate song-date combinations:",
    len(song_date_duplicates))

Duplicate song-date combinations: 50


In [11]:
print("Unique songs:", df["song"].nunique())
print("Unique artists:", df["artist"].nunique())
print("Unique dates:", df["date"].nunique())


Unique songs: 943
Unique artists: 297
Unique dates: 555


In [12]:
print(df["album_type"].value_counts())
print(df["is_explicit"].value_counts())

album_type
album          19742
single          8016
compilation       42
Name: count, dtype: int64
is_explicit
False    14465
True     13335
Name: count, dtype: int64


In [13]:
print(df["popularity"].min())
print(df["popularity"].max())
print(df["popularity"].mean())

0
100
87.66561151079136


In [20]:
rows_per_date = df.groupby("date").size()

print(rows_per_date.value_counts().sort_index())

dates_not_50 = rows_per_date[rows_per_date != 50]

print(dates_not_50)

50     554
100      1
Name: count, dtype: int64
date
2025-03-01    100
dtype: int64


## 5. Data Cleaning

In [21]:
df_clean = df.drop_duplicates().copy()

print("Original rows:", len(df))
print("Rows after removing exact duplicates:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Original rows: 27800
Rows after removing exact duplicates: 27790
Rows removed: 10


In [25]:
march_1 = df_clean[df_clean["date"] == "2025-03-01"]

print("Rows on 2025-03-01:", len(march_1))
print("Unique positions:", march_1["position"].nunique())
print("Duplicate positions:",
    march_1["position"].duplicated().sum())

Rows on 2025-03-01: 90
Unique positions: 50
Duplicate positions: 40


In [37]:
df_clean = df.drop_duplicates().copy()

df_analysis = df_clean[
    df_clean["date"] != pd.Timestamp("2025-03-01")
].copy()

print("Original rows:", len(df))
print("After exact duplicate removal:", len(df_clean))
print("Final analysis rows:", len(df_analysis))

Original rows: 27800
After exact duplicate removal: 27790
Final analysis rows: 27700


## 6. Final Validation

In [38]:
rows_per_date = df_analysis.groupby("date").size()

print("Number of dates:", df_analysis["date"].nunique())
print("Rows per date:")
print(rows_per_date.value_counts().sort_index())

print("\nInvalid positions:",
      (~df_analysis["position"].between(1, 50)).sum())

print("Missing values:",
      df_analysis.isnull().sum().sum())

print("Duplicate rows:",
      df_analysis.duplicated().sum())

Number of dates: 554
Rows per date:
50    554
Name: count, dtype: int64

Invalid positions: 0
Missing values: 0
Duplicate rows: 0


## 7. Export Processed Dataset

In [40]:
OUTPUT_PATH = "../data/processed/us_top50_clean.csv"

df_analysis.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Processed dataset saved to: {OUTPUT_PATH}")
print(f"Rows saved: {len(df_analysis):,}")

Processed dataset saved to: ../data/processed/us_top50_clean.csv
Rows saved: 27,700
